# Hands-on 1: Code Optimizations for Energy Efficiency

This notebook explores three code optimization techniques that reduce both execution time and energy consumption of GPU kernels:

1. **Thread block size tuning**: find the block configuration that best utilizes the GPU's hardware resources
2. **Work Per Thread (WPT)**: assign multiple data elements to each thread to improve memory access efficiency
3. **Kernel fusion**: merge multiple kernels into one to eliminate redundant global memory traffic

We use [Kernel Tuner](https://kerneltuner.github.io) to automate the search and [NVML](https://developer.nvidia.com/management-library-nvml) to measure GPU energy consumption.


## Installing Dependencies

In [ ]:
%%capture
%pip install kernel-tuner

import numpy as np
import kernel_tuner as kt
from kernel_tuner.observers.nvml import NVMLObserver
from kernel_tuner.util import get_best_config

# helper function to find and print measured execution time and energy consumption
def find_best(*all_results):
  time, energy = 0, 0
  for results in all_results:
    best = get_best_config(results, "time")
    time += best["time"]
    energy += best["energy"]
  return time, energy

def print_best(*all_results):
  time, energy = find_best(*all_results)
  print(f"\nExecution time:\t{time} ms\nEnergy:\t\t{energy} Joule\n")


In [ ]:
# the size of the vectors
size = 150_000_000

# all the kernel input and output data need to use numpy data types,
# note that we explicitly state that these arrays should consist of
# 32 bit floating-point values, to match our kernel source code
a = np.random.randn(size).astype(np.float32)
b = np.random.randn(size).astype(np.float32)
c = np.zeros_like(b)
a_scaled = np.zeros_like(a)
factor = np.float32(3.14)
n = np.int32(size)

# setup Kernel Tuner's NVMLObserver to measure energy
nvmlobserver = NVMLObserver(["nvml_energy"])
metrics = dict()
metrics["energy"] = lambda p:p["nvml_energy"]

options = dict(
    metrics=metrics,
    observers=[nvmlobserver],
)

## Establish a Baseline

To understand how to achieve higher energy-efficiency for a GPU application, we first establish a baseline using a common workload: a scaled vector addition, represented as `c = a * factor + b`.

This operation is initially implemented using two distinct CUDA kernels, which allows us to identify potential areas for optimization later on.

These two kernels are:

-   `scale_vector`: This kernel computes the intermediate result `a_scaled = a * factor`.
-   `vector_add`: This kernel then completes the operation by computing `c = a_scaled + b`.

In [ ]:
%%writefile scale_vector.cu

__global__ void scale_vector(float * a, float * a_scaled, float factor, int n) {
    int i = (blockIdx.x * blockDim.x) + threadIdx.x;
    if (i < n) {
        a_scaled[i] = a[i] * factor;
    }
}

In [ ]:
%%writefile vector_add.cu

__global__ void vector_add(float * a, float * b, float * c, int n) {
    int i = (blockIdx.x * blockDim.x) + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

Before we can optimize, we need a clear point of reference. In this first step, we establish a baseline for both execution time and energy consumption of our `scale_vector` and `vector_add` kernels.

We will run these kernels with a fixed, unoptimized thread block size of 32 threads per block. This initial measurement will serve as the benchmark against which all subsequent optimizations will be compared, allowing us to quantify the improvements in energy efficiency and performance.

In [ ]:
reference_scale, _ = kt.tune_kernel(
    "scale_vector", "scale_vector.cu", size,
    [a, a_scaled, factor, n],
    {"block_size_x": [32]},
    answer=[None, a * factor, None, None],
    **options,
)

reference_sum, _ = kt.tune_kernel(
    "vector_add", "vector_add.cu", size,
    [a * factor, b, c, n],
    {"block_size_x": [32]},
    answer=[None, None, a * factor + b, None],
    **options,
)

print_best(reference_scale, reference_sum)

## Optimization 1: Thread Block Size Tuning

The choice of thread block size (the number of threads executed concurrently within a block) is crucial for both performance and energy efficiency on a GPU. It directly impacts:

*   **Resource Utilization:** An optimal thread block size ensures that the GPU's streaming multiprocessors (SMs) are fully utilized, preventing idle resources and maximizing throughput.
*   **Memory Access Patterns:** It can affect how data is accessed in shared memory and global memory, influencing cache hit rates and reducing latency.
*   **Coarse-grain vs. Fine-grain Parallelism:** An appropriate balance is needed to effectively distribute work among SMs while ensuring enough parallelism within each SM.

The ideal block size is hardware-dependent and often algorithm-specific, making empirical measurement essential. We cannot predict it from first principles, so tuning is necessary.

### **Assignment**

Modify the `tune_params["block_size_x"]` list in the next cell to include a range of thread block sizes (e.g., powers of 2 like `[32, 64, 128, 256, 512, 1024]`). Then, execute the cell to measure the execution time and energy consumption for each configuration, and identify the best one.

In [ ]:
# add some possible thread-block sizes to the tunable parameters
tune_params = dict()
tune_params["block_size_x"] = [32]

# call tune_kernel to start the tuning process for scale_vector
tuned_scale, _ = kt.tune_kernel(
    "scale_vector", "scale_vector.cu", size,
    [a, a_scaled, factor, n],
    tune_params,
    answer=[None, a * factor, None, None],
    **options,
)

# call tune_kernel to start the tuning process for vector_add
tuned_sum, _ = kt.tune_kernel(
    "vector_add", "vector_add.cu", size,
    [a * factor, b, c, n],
    tune_params,
    answer=[None, None, a * factor + b, None],
    **options,
)

print_best(tuned_scale, tuned_sum)

## Optimization 2: Work Per Thread (WPT)

Work Per Thread (WPT) is another interesting optimization technique for GPU kernels. With a WPT of 1, each thread processes a single array element. By increasing the WPT, we assign each thread to process multiple elements in a strided loop. In general, this can significantly improve performance and energy efficiency by:

*   **Increasing Arithmetic Intensity:** Each thread performs more computations relative to its memory accesses, keeping the ALUs busy.
*   **Improving Memory Coalescing:** Accessing multiple elements in a strided fashion can lead to better memory coalescing, where multiple memory requests from threads within a warp are combined into a single, efficient transaction.
*   **Reducing Kernel Launch Overhead:** By doing more work per thread, the total number of threads required for the entire computation decreases, which can reduce the overhead associated with launching kernels.

The following two cells have this optimization already implemented in the ``scale_vector`` and ``vector_add`` kernels.

In [ ]:
%%writefile scale_vector_opt.cu

__global__ void scale_vector(float * a, float * a_scaled, float factor, int n) {
    int i = (blockIdx.x * blockDim.x * wpt) + threadIdx.x;
    #pragma unroll
    for (int offset = 0; offset < wpt; offset++) {
        int item = i + offset * block_size_x;
        if (item < n) {
            a_scaled[item] = a[item] * factor;
        }
    }
}

In [ ]:
%%writefile vector_add_opt.cu

__global__ void vector_add(float * a, float * b, float * c, int n) {
    int i = (blockIdx.x * blockDim.x * wpt) + threadIdx.x;
    #pragma unroll
    for (int offset = 0; offset < wpt; offset++) {
        int item = i + offset * block_size_x;
        if (item < n) {
            c[item] = a[item] + b[item];
        }
    }
}

### **Assignment**

Extend the `tune_params` dictionary in the next cell to include `wpt` values. For example, you can use powers of 2 like `[1, 2, 4, 8, 16]`. Execute the cell to tune for both `block_size_x` and `wpt` simultaneously, and observe the further improvements in execution time and energy consumption.

In [ ]:
# add the values for the wpt parameter
tune_params["wpt"] = [1]

# call tune_kernel to start the tuning process for scale_vector
opt_scale, _ = kt.tune_kernel(
    "scale_vector", "scale_vector_opt.cu", size,
    [a, a_scaled, factor, n],
    tune_params,
    grid_div_x=["block_size_x", "wpt"],
    answer=[None, a * factor, None, None],
    **options,
)

# call tune_kernel to start the tuning process for vector_add
opt_sum, _ = kt.tune_kernel(
    "vector_add", "vector_add_opt.cu", size,
    [a * factor, b, c, n],
    tune_params,
    grid_div_x=["block_size_x", "wpt"],
    answer=[None, None, a * factor + b, None],
    **options,
)

print_best(opt_scale, opt_sum)

## Optimization 3: Kernel Fusion

When we have multiple kernel calls, as in our baseline example with `scale_vector` and `vector_add`, an intermediate array (`a_scaled`) is often created. This array is written to global memory by the first kernel and then read back by the second. This round-trip through global memory (DRAM) is inherently wasteful, consuming significant execution time and energy.

For memory-bound kernels, this redundant global memory traffic can account for a substantial fraction of both execution time and energy consumption. By merging both operations into a single **SAXPY** kernel (*Single-precision A·X Plus Y*), we can eliminate this intermediate array entirely.

The fused SAXPY kernel reads `a` and `b` once from global memory and writes the final result `c` once. This reduces the number of global memory operations, leading to significant performance and energy efficiency gains.

### **Assignment**

Implement the body of the fused `saxpy` CUDA kernel.

In [ ]:
%%writefile saxpy.cu

__global__ void saxpy(float * a, float * b, float * c, float factor, int n) {
    // TODO: Implement the saxpy function
    // Hint: Compute c as c = a * factor + b
    // Hint2: Also implement a work-per-thread loop, like in vector_add_opt
}

In [ ]:
# If you did not implement a work-per-thread loop in the saxpy kernel
# make sure to uncomment the next line:
# tune_params["wpt"] = [1]

results_saxpy, _ = kt.tune_kernel(
    "saxpy", "saxpy.cu", size,
    [a, b, c, factor, n],
    tune_params,
    grid_div_x=["block_size_x", "wpt"],
    answer=[None, None, a * factor + b, None, None],
    **options,
)

print_best(results_saxpy)

## Results

The table below presents a comparative analysis of the best configurations found at each stage of optimization, illustrating the cumulative benefits of applying thread block size tuning, Work Per Thread (WPT) optimization, and kernel fusion.

For each stage, you will see the best achieved execution time and energy consumption. The 'Speedup' and 'Energy saving (%)' columns show the improvement relative to the initial untuned baseline. This provides a clear quantitative measure of how each optimization contributes to more energy-efficient GPU computing.

In [ ]:
import pandas as pd

data = [
    ("Reference",              *find_best(reference_scale, reference_sum)),
    ("Tuned block size",       *find_best(tuned_scale, tuned_sum)),
    ("Tuned block size + WPT", *find_best(opt_scale, opt_sum)),
    ("SAXPY (fused)",          *find_best(results_saxpy)),
]

df = pd.DataFrame(data, columns=["Stage", "Time (ms)", "Energy (J)"])
df = df.set_index("Stage")

ref_time   = df["Time (ms)"]["Reference"]
ref_energy = df["Energy (J)"]["Reference"]
df["Speedup"]           = (ref_time   / df["Time (ms)"]).round(2)
df["Energy saving (%)"] = ((1 - df["Energy (J)"] / ref_energy) * 100).round(1)

df.round(3)
